# 🎯 HireIQ — AI-Powered Resume Screening & Candidate Ranking System
## FUTURE_ML_03 | NLP + Cosine Similarity | HR-Tech Internship Project

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python)](https://python.org)
[![scikit-learn](https://img.shields.io/badge/scikit--learn-1.3-orange)](https://scikit-learn.org)
[![Status](https://img.shields.io/badge/Status-Complete-brightgreen)]()

---

**Company:** NovaTech Analytics — 200-person SaaS company hiring a Data Analyst  
**Problem:** Recruiting team receives 200+ applications per role; manual screening takes 3–5 days and is inconsistent  
**Solution:** HireIQ reads every resume, scores it against the job description, extracts matched/missing skills, and produces a ranked shortlist in seconds

| | |
|---|---|
| **Dataset** | 20 realistic candidate resumes + 1 job description |
| **Method** | TF-IDF Vectorisation + Cosine Similarity + Weighted Skill Scoring |
| **Outputs** | Ranked table, skill heatmap, 7 professional charts |
| **Best Candidate** | Aisha Patel — Score 55.0% (Strong Fit, 13/14 skills matched) |
| **Author** | \[Your Name\] |

---


## Section A — Import Libraries

In [ ]:
# ── Standard libraries ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re, string, warnings, os
from collections import Counter
warnings.filterwarnings('ignore')

# ── Machine Learning ──────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Visual style ──────────────────────────────────────────────────────────
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
BLUE   = '#1A73E8'
GREEN  = '#34A853'
RED    = '#EA4335'
AMBER  = '#FBBC04'
PALETTE = [BLUE, GREEN, AMBER, RED, '#9334E6', '#00ACC1', '#FF6D00', '#795548']

print("✅ All libraries imported.")


## Section B — Load Resume Dataset

**Dataset:** 20 realistic candidate resumes in `resumes.csv` + `job_description.txt`

> **To use the real Kaggle Resume Dataset:**  
> Download from: https://www.kaggle.com/datasets/gauravduttakiit/resume-dataset  
> Then: `df = pd.read_csv('../dataset/resume_dataset.csv')`  
> Map the `Resume` column to `resume_text` and `Category` column to candidate labels.


In [ ]:
# ── Load resumes ──────────────────────────────────────────────────────────
df = pd.read_csv('../dataset/resumes.csv')
print(f"✅ Loaded {len(df)} resumes")
print(f"   Columns: {list(df.columns)}")
df.head()


## Section C — Load Job Description

In [ ]:
# ── Load job description ──────────────────────────────────────────────────
with open('../dataset/job_description.txt', 'r') as f:
    jd_raw = f.read()

print("✅ Job Description loaded:")
print("─" * 50)
print(jd_raw[:600] + "...")


## Section D — Text Cleaning

We apply a 6-step NLP cleaning pipeline to both resumes and the job description before comparison.

| Step | Action | Example |
|---|---|---|
| 1 | Lowercase | `Python` → `python` |
| 2 | Remove URLs / emails / numbers | `john@mail.com` → `` |
| 3 | Remove punctuation | `C++,` → `c` |
| 4 | Tokenise | Split on whitespace |
| 5 | Remove stopwords | Drop `the`, `and`, `of`… |
| 6 | Filter short tokens | Keep tokens with len > 1 |


In [ ]:
# ── Stopword list (no NLTK download required) ─────────────────────────────
STOPWORDS = set(['i','me','my','we','our','you','your','he','him','his','she','her',
'it','its','they','them','their','this','that','these','those','am','is','are','was',
'were','be','been','have','has','had','do','does','did','a','an','the','and','but',
'if','or','as','of','at','by','for','with','about','to','from','in','on','not','no',
'so','than','too','very','can','will','just','should','now','also','such','more',
'most','other','some','any','all','both'])

def clean_text(text: str) -> str:
    """NLP cleaning pipeline — lowercase, remove noise, tokenise, filter stopwords."""
    text = text.lower()
    text = re.sub(r'http\S+|\S+@\S+|\d+', ' ', text)
    text = text.translate(str.maketrans(string.punctuation, ' '*len(string.punctuation)))
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

# Apply to all resumes and JD
df['clean_resume'] = df['resume_text'].apply(clean_text)
jd_clean           = clean_text(jd_raw)

# Show a cleaning example
sample = df.iloc[0]
print("── Cleaning Example ──────────────────────────────────────────")
print(f"BEFORE: {sample['resume_text'][:120]}")
print(f"AFTER : {sample['clean_resume'][:120]}")
print(f"\n✅ Cleaned {len(df)} resumes + 1 job description")


## Section E — Exploratory Analysis

In [ ]:
# ── Skill taxonomy — weighted by importance to this role ──────────────────
SKILLS = {
    'Python':             {'weight': 1.5, 'keywords': ['python']},
    'SQL':                {'weight': 1.4, 'keywords': ['sql']},
    'Machine Learning':   {'weight': 1.4, 'keywords': ['machine learning','ml','scikit-learn']},
    'Power BI':           {'weight': 1.3, 'keywords': ['power bi','powerbi']},
    'Tableau':            {'weight': 1.2, 'keywords': ['tableau']},
    'Excel':              {'weight': 1.1, 'keywords': ['excel','google sheets']},
    'Statistics':         {'weight': 1.2, 'keywords': ['statistics','regression','predictive']},
    'Communication':      {'weight': 1.1, 'keywords': ['communication','presentation']},
    'Project Management': {'weight': 1.1, 'keywords': ['project management']},
    'Cloud':              {'weight': 1.2, 'keywords': ['cloud','aws','gcp','azure']},
    'Git':                {'weight': 1.1, 'keywords': ['git','github']},
    'ETL':                {'weight': 1.1, 'keywords': ['etl','pipeline','wrangling','data cleaning']},
    'Data Visualisation': {'weight': 1.1, 'keywords': ['visualization','dashboard','reporting']},
    'R':                  {'weight': 0.9, 'keywords': [' r ','r programming']},
}

def extract_skills(text: str):
    """Keyword-based skill extraction. Returns (matched, missing) lists."""
    t = text.lower()
    found   = [s for s, m in SKILLS.items() if any(kw in t for kw in m['keywords'])]
    missing = [s for s in SKILLS if s not in found]
    return found, missing

# Count skill frequency across all resumes
all_matched = []
for text in df['resume_text']:
    found, _ = extract_skills(text)
    all_matched.extend(found)

skill_counts = Counter(all_matched)
print("Skills found across all resumes:")
for skill, count in sorted(skill_counts.items(), key=lambda x: -x[1]):
    bar = '█' * count
    print(f"  {skill:<22} {bar} {count}")


In [ ]:
os.makedirs('../outputs', exist_ok=True)

# ── Chart 3: Most Common Skills ───────────────────────────────────────────
skills_sorted = sorted(skill_counts.items(), key=lambda x: x[1], reverse=True)
skill_names, skill_vals = zip(*skills_sorted)
color_bars = [GREEN if v >= 15 else BLUE if v >= 10 else '#90CAF9' for v in skill_vals]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(skill_names, skill_vals, color=color_bars, width=0.65)
for bar, val in zip(bars, skill_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
            str(val), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Most Common Skills Across All Candidate Resumes', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Candidates with Skill')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('../outputs/03_common_skills.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: 03_common_skills.png")


## Section F — Skill Extraction & Weighted Scoring

Each candidate receives four sub-scores, then a weighted composite:

| Component | Weight | What it measures |
|---|---|---|
| **TF-IDF Cosine Similarity** | 40% | Overall text similarity to the JD |
| **Weighted Skill Match** | 35% | Skills present, weighted by role importance |
| **Experience Score** | 15% | Years/seniority keywords detected |
| **Education Score** | 10% | Degree level detected |

**Why weighted skills?** Python and SQL carry 1.5× and 1.4× weight respectively — they are non-negotiable for this Data Analyst role. R carries only 0.9× as it's optional.


In [ ]:
# ── Experience and education keyword scoring ──────────────────────────────
EXPERIENCE_KW = ['years experience','year experience','years of experience',
                  'senior','lead','manager','principal','phd','master','bachelor']
EDUCATION_KW  = {'phd': 4, 'master': 3, 'bachelor': 2, 'degree': 1}

def experience_score(text: str) -> float:
    """Returns 0–1 based on seniority/experience keywords."""
    t = text.lower()
    hits = sum(1 for kw in EXPERIENCE_KW if kw in t)
    return min(hits / 3, 1.0)

def education_score(text: str) -> float:
    """Returns 0–1 based on highest education keyword found."""
    t = text.lower()
    score = max((pts for kw, pts in EDUCATION_KW.items() if kw in t), default=0)
    return score / 4

# Demo on first candidate
sample_text = df.iloc[0]['resume_text']
found, missing = extract_skills(sample_text)
print(f"Candidate: {df.iloc[0]['candidate_name']}")
print(f"  Matched skills  : {found}")
print(f"  Missing skills  : {missing}")
print(f"  Experience score: {experience_score(sample_text):.2f}")
print(f"  Education score : {education_score(sample_text):.2f}")


## Section G — TF-IDF Vectorisation

In [ ]:
# ── TF-IDF on resumes + job description ──────────────────────────────────
# We include the JD in the corpus so all texts share the same vocabulary space

corpus = list(df['clean_resume']) + [jd_clean]

tfidf = TfidfVectorizer(
    max_features=3000,    # top 3,000 terms
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    # log-scale TF dampening
    min_df=1
)
tfidf_matrix = tfidf.fit_transform(corpus)

# Separate resume vectors from JD vector
resume_vecs = tfidf_matrix[:-1]
jd_vec      = tfidf_matrix[-1]

print(f"✅ TF-IDF matrix: {tfidf_matrix.shape}")
print(f"   Features (vocabulary): {tfidf_matrix.shape[1]:,}")
print(f"   Resume vectors:        {resume_vecs.shape[0]}")
print(f"   JD vector:             1 × {jd_vec.shape[1]}")


## Section H — Cosine Similarity Scoring

**Cosine Similarity** measures the angle between two TF-IDF vectors.  
A score of 1.0 = identical text. A score of 0.0 = no shared vocabulary.

It is ideal for resume-JD matching because it is length-agnostic — a short resume and a long one are compared fairly on vocabulary overlap, not word count.


In [ ]:
# ── Compute cosine similarity for all resumes vs JD ──────────────────────
cosine_scores = cosine_similarity(resume_vecs, jd_vec).flatten()

print("Cosine Similarity Scores (text match vs Job Description):")
print(f"{'─'*45}")
for name, score in zip(df['candidate_name'], cosine_scores):
    bar = '█' * int(score * 40)
    print(f"  {name:<25} {score:.4f}  {bar}")

print(f"\n  Max : {cosine_scores.max():.4f} ({df.iloc[cosine_scores.argmax()]['candidate_name']})")
print(f"  Min : {cosine_scores.min():.4f} ({df.iloc[cosine_scores.argmin()]['candidate_name']})")
print(f"  Mean: {cosine_scores.mean():.4f}")


## Section I — Candidate Ranking (Composite Score)

In [ ]:
# ── Build composite scores ────────────────────────────────────────────────
max_skill_weight = sum(m['weight'] for m in SKILLS.values())

results = []
for i, row in df.iterrows():
    text        = row['resume_text']
    found, miss = extract_skills(text)
    exp_sc      = experience_score(text)
    edu_sc      = education_score(text)
    cos_sc      = cosine_scores[i]

    # Weighted skill score
    skill_raw   = sum(SKILLS[s]['weight'] for s in found)
    skill_pct   = skill_raw / max_skill_weight

    # Composite (weighted sum)
    composite   = (0.40 * cos_sc + 0.35 * skill_pct +
                   0.15 * exp_sc + 0.10 * edu_sc)
    composite   = round(composite * 100, 1)

    fit = '🟢 Strong Fit' if composite >= 55 else '🟡 Good Fit' if composite >= 40 else '🔴 Weak Fit'

    results.append({
        'Candidate':      row['candidate_name'],
        'Cosine_Score':   round(cos_sc * 100, 1),
        'Skill_Score':    round(skill_pct * 100, 1),
        'Exp_Score':      round(exp_sc * 100, 1),
        'Edu_Score':      round(edu_sc * 100, 1),
        'Final_Score':    composite,
        'Matched_Skills': ', '.join(found),
        'Missing_Skills': ', '.join(miss),
        'Num_Matched':    len(found),
        'Fit_Label':      fit,
    })

results_df = (pd.DataFrame(results)
                .sort_values('Final_Score', ascending=False)
                .reset_index(drop=True))
results_df['Rank'] = results_df.index + 1

# Save full ranking table
results_df.to_csv('../outputs/candidate_rankings.csv', index=False)
print("✅ Ranking table saved to ../outputs/candidate_rankings.csv")
print()

# Display clean summary
display_cols = ['Rank','Candidate','Final_Score','Num_Matched','Fit_Label']
results_df[display_cols]


## Section J — Visualisations

In [ ]:
# ── Chart 1: Top 10 Candidates ────────────────────────────────────────────
top10 = results_df.head(10)
colors = [GREEN if s>=55 else AMBER if s>=40 else RED for s in top10['Final_Score']]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(top10['Candidate'][::-1], top10['Final_Score'][::-1],
               color=colors[::-1], height=0.6)
ax.axvline(55, color=GREEN, linestyle='--', alpha=0.5, linewidth=1.3)
ax.axvline(40, color=AMBER, linestyle='--', alpha=0.5, linewidth=1.3)
for bar, val in zip(bars, top10['Final_Score'][::-1]):
    ax.text(val+0.4, bar.get_y()+bar.get_height()/2,
            f'{val}%', va='center', fontsize=10, fontweight='bold')

patches = [mpatches.Patch(color=GREEN,label='Strong Fit (≥55%)'),
           mpatches.Patch(color=AMBER,label='Good Fit (≥40%)'),
           mpatches.Patch(color=RED,  label='Weak Fit (<40%)')]
ax.legend(handles=patches, loc='lower right', fontsize=10)
ax.set_title('Top 10 Candidates — Composite Match Score\nNovaTech Analytics | Data Analyst Role',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Composite Match Score (%)')
ax.set_xlim(0, 80)
plt.tight_layout()
plt.savefig('../outputs/01_top10_candidates.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: 01_top10_candidates.png")


In [ ]:
# ── Chart 2: Score Distribution + Component Averages ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(results_df['Final_Score'], bins=10, color=BLUE, alpha=0.8, edgecolor='white')
axes[0].axvline(results_df['Final_Score'].mean(), color=RED, linestyle='--', linewidth=2,
                label=f"Mean: {results_df['Final_Score'].mean():.1f}%")
axes[0].set_title('Score Distribution — All 20 Candidates', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Composite Score (%)'); axes[0].set_ylabel('Candidates')
axes[0].legend(fontsize=11)

comp_means = results_df[['Cosine_Score','Skill_Score','Exp_Score','Edu_Score']].mean()
axes[1].bar(['TF-IDF\nSimilarity','Skill\nMatch','Experience','Education'],
            comp_means.values, color=PALETTE[:4], width=0.55)
axes[1].set_title('Average Score by Component', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Average Score (%)')
for i, val in enumerate(comp_means.values):
    axes[1].text(i, val+0.4, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/02_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: 02_score_distribution.png")


In [ ]:
# ── Chart 4: Missing Skills Frequency ────────────────────────────────────
all_missing = []
for _, row in results_df.iterrows():
    if row['Missing_Skills']:
        all_missing.extend(row['Missing_Skills'].split(', '))
miss_counts = Counter(all_missing)
miss_sorted = sorted(miss_counts.items(), key=lambda x: x[1], reverse=True)
miss_names, miss_vals = zip(*miss_sorted)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(miss_names[::-1], miss_vals[::-1], color=RED, alpha=0.8, height=0.6)
for bar, val in zip(bars, miss_vals[::-1]):
    ax.text(val+0.1, bar.get_y()+bar.get_height()/2, str(val), va='center', fontsize=10, fontweight='bold')
ax.set_title('Most Frequently Missing Skills — Candidate Pool', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Candidates Missing This Skill')
plt.tight_layout()
plt.savefig('../outputs/04_missing_skills.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: 04_missing_skills.png")


In [ ]:
# ── Chart 5: Skill Coverage Heatmap — Top 15 ─────────────────────────────
skill_list = list(SKILLS.keys())
heatmap_data = []
for _, row in results_df.head(15).iterrows():
    matched = row['Matched_Skills'].split(', ') if row['Matched_Skills'] else []
    heatmap_data.append([1 if s in matched else 0 for s in skill_list])

hm_df = pd.DataFrame(heatmap_data,
                      index=results_df.head(15)['Candidate'],
                      columns=skill_list)
fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(hm_df, cmap='RdYlGn', annot=True, fmt='d', linewidths=0.5,
            linecolor='white', cbar=False, ax=ax, annot_kws={'size': 9},
            xticklabels=[s.replace(' ', '\n') for s in skill_list])
ax.set_title('Skill Coverage Heatmap — Top 15 Candidates\n(Green=Present  Red=Missing)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Required Skills', fontsize=11); ax.set_ylabel('Candidate', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9); plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/05_skill_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: 05_skill_heatmap.png — most impressive chart for portfolio!")


In [ ]:
# ── Chart 6: Fit Category Breakdown ──────────────────────────────────────
fit_counts  = results_df['Fit_Label'].value_counts()
clean_labels = [l.split(' ',1)[1] for l in fit_counts.index]
colors_fit   = [GREEN if 'Strong' in l else AMBER if 'Good' in l else RED for l in fit_counts.index]

fig, ax = plt.subplots(figsize=(7, 5))
ax.pie(fit_counts.values, labels=clean_labels, autopct='%1.0f%%',
       colors=colors_fit, startangle=140, pctdistance=0.72,
       wedgeprops={'edgecolor':'white','linewidth':2.5})
ax.set_title('Candidate Fit Distribution\nData Analyst Role — NovaTech Analytics',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/06_fit_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: 06_fit_distribution.png")


In [ ]:
# ── Chart 7: Score Components — Top 10 ───────────────────────────────────
top10 = results_df.head(10)
x, w = np.arange(len(top10)), 0.18

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - 1.5*w, top10['Cosine_Score'], w, label='TF-IDF Similarity', color=BLUE)
ax.bar(x - 0.5*w, top10['Skill_Score'],  w, label='Skill Match',       color=GREEN)
ax.bar(x + 0.5*w, top10['Exp_Score'],    w, label='Experience',        color=AMBER)
ax.bar(x + 1.5*w, top10['Edu_Score'],    w, label='Education',         color='#9334E6')
ax.set_xticks(x)
ax.set_xticklabels([n.split()[0] for n in top10['Candidate']], rotation=35, ha='right')
ax.set_ylabel('Component Score (%)'); ax.legend(fontsize=10)
ax.set_title('Score Component Breakdown — Top 10 Candidates', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/07_score_components.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved: 07_score_components.png")


## Section K — Prediction Demo (Final Output)

In [ ]:
print("\n" + "="*70)
print("  🎯 HireIQ — Candidate Screening Results")
print("  Role: Data Analyst  |  Company: NovaTech Analytics")
print("  Candidates evaluated: 20  |  Screened in: < 2 seconds")
print("="*70)

for _, row in results_df.iterrows():
    icon = '🟢' if row['Fit_Label'].startswith('🟢') else '🟡' if row['Fit_Label'].startswith('🟡') else '🔴'
    label = row['Fit_Label'].split(' ',1)[1]

    print(f"\n  Rank #{row['Rank']:>2}  {row['Candidate']}")
    print(f"  {icon} {label}  |  Score: {row['Final_Score']}%")
    print(f"  ✅ Matched ({row['Num_Matched']}): {row['Matched_Skills'] or 'None'}")
    print(f"  ❌ Missing ({row['Num_Missing']}): {row['Missing_Skills'] or 'None'}")

print("\n" + "="*70)
print(f"  ✅ Top shortlist (Strong + Good Fit): "
      f"{(results_df['Final_Score']>=40).sum()} of {len(results_df)} candidates")
print("="*70)


## Scoring Logic Explained

### How scores are calculated

The **Composite Score** is a weighted average of four components:

```
Composite = (0.40 × TF-IDF Cosine) + (0.35 × Weighted Skill Match)
          + (0.15 × Experience Score) + (0.10 × Education Score)
```

**TF-IDF Cosine Similarity (40%)**  
Converts both the resume and JD into vectors of 3,000 term-frequency features. Cosine similarity measures the angle between the two vectors — a high score means the candidate uses the same professional vocabulary as the job description.

**Weighted Skill Match (35%)**  
Each required skill has an importance weight. Python = 1.5×, SQL = 1.4×, Machine Learning = 1.4×, Cloud = 1.2×, etc. A candidate who matches all high-weight skills scores higher than one who matches an equal number of low-weight skills.

**Experience Score (15%)**  
Presence of keywords like `years experience`, `senior`, `lead`, `master`, `phd` — each adds to the experience score, capped at 1.0 after 3 or more signals.

**Education Score (10%)**  
Detected degree level: PhD = 1.0, Master = 0.75, Bachelor = 0.5, Degree mention = 0.25.

### Why some candidates rank higher
A candidate with Python, SQL, ML, and Power BI ranks above one with only Excel and SQL — because weighted skill coverage accounts for the business priority of each skill, not just count.

### ATS Compatibility Note
This system mirrors how commercial ATS tools (Workday, Greenhouse, Lever) work: keyword matching + similarity scoring. Candidates should ensure their resume text uses the exact terminology from job descriptions.

---

## Ethics & Fairness Considerations

- **Names are not used in scoring** — scores are based purely on text content
- **Weighted skills are transparent** — weights are documented and explainable
- **Human-in-the-loop** — all shortlisted candidates should be reviewed by a human recruiter before decisions are made
- **Bias auditing** — periodically review shortlists for demographic patterns if candidate metadata is available
- **Skill gap ≠ rejection** — missing skills should be treated as development opportunities, not disqualifiers, especially for junior roles

---

## Deployment Suggestions

1. **Streamlit web app** — drag-and-drop PDF resumes, paste JD, get ranked table instantly
2. **FastAPI endpoint** — `/screen` POST endpoint for integration with ATS systems
3. **Google Forms + Sheets** — non-technical recruiters submit resumes via form, results auto-populate in Sheets
4. **Email parser** — auto-screen resumes sent to a dedicated HR mailbox

---
*FUTURE_ML_03 | HireIQ | Internship Portfolio Project*
